# einops-rearrange-flatten — ex2: unflatten a linear-projected vector back to a CNN feature map

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-rearrange-flatten`. Running the final beacon cell reports progress against the `Einops: Rearrange-as-flatten` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Rearrange-as-flatten` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-rearrange-flatten`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-rearrange-flatten"
DD_SUBTOPIC = "Einops: Rearrange-as-flatten"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## einops.rearrange — flatten and unflatten

Parenthesised axes in einops can do *composition* (flatten) **or** *decomposition* (unflatten), depending on which side of the arrow they appear on. To unflatten you must supply the named sizes:

```python
x = rearrange(flat, 'b (c h w) -> b c h w', c=C, h=H, w=W)
```

**This drill (ex2) vs ex1.** ex1 did the **forward** flatten `'b c h w -> b (c h w)'` (CNN feature map → linear-head input). ex2 does the **inverse** un-flatten `'b (c h w) -> b c h w'` — the move you need going from a linear layer's flat output BACK to a spatial feature map (e.g. the first stage of a decoder / generator).

### Exercise 2 — unflatten a linear-projected vector back to a CNN feature map

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `einops.rearrange` with the `'b (c h w) -> b c h w'` pattern to decompose a 2-D `(B, C*H*W)` linear-projection output back into a 4-D `(B, C, H, W)` CNN feature map, supplying the named axis sizes.
> Keywords: unflatten, decoder, rearrange, axis-decomposition
> ```

**KCs targeted:** `rearrange-axis-decomposition-via-parens`, `rearrange-named-sizes-required`

Implement `ex2_unflatten_to_feature_map(flat, C, H, W)`.

A `Linear` layer produced a 2-D batch of vectors `flat` of shape `(B, C*H*W)`. Reshape it into a 4-D feature map `(B, C, H, W)` using the einops decomposition pattern.

**Rules.**
1. Use `einops.rearrange` with the `'b (c h w) -> b c h w'` pattern.
2. You **must** pass `c=C, h=H, w=W` as keyword args — einops can't infer how to split the composite axis without them.
3. The mapping must preserve the row-major ordering: the leftmost axis in the parenthesis (`c`) varies SLOWEST, the rightmost (`w`) varies FASTEST. Equivalently, `flat[b, c*H*W + h*W + w] == out[b, c, h, w]`.

Inputs:
- `flat`: `(B, C*H*W)` float tensor.
- `C`, `H`, `W`: int axis sizes.

Output: `(B, C, H, W)` float tensor.

In [ ]:
def ex2_unflatten_to_feature_map(flat: Tensor, C: int, H: int, W: int) -> Tensor:
    """Decompose (B, C*H*W) → (B, C, H, W) via einops rearrange."""
    raise NotImplementedError()


def _test_ex2():
    # Build a deterministic flat vector then unflatten.
    B, C, H, W = 2, 3, 4, 5
    flat = t.arange(B * C * H * W, dtype=t.float32).reshape(B, C * H * W)
    out = ex2_unflatten_to_feature_map(flat, C, H, W)
    assert out.shape == (B, C, H, W), f'shape {tuple(out.shape)} != {(B,C,H,W)}'
    assert out.dtype == t.float32, f'dtype {out.dtype}'

    # Verify row-major mapping cell-by-cell.
    for b in range(B):
        for c in range(C):
            for h in range(H):
                for w in range(W):
                    expected = flat[b, c * H * W + h * W + w].item()
                    got = out[b, c, h, w].item()
                    assert got == expected, f'cell ({b},{c},{h},{w}): got {got}, expected {expected}'

    # Round-trip identity: unflatten then flatten back returns the original.
    round_trip = einops.rearrange(out, 'b c h w -> b (c h w)')
    assert t.equal(round_trip, flat), 'round-trip unflatten→flatten failed'

    # Different shape — non-square decoder use-case (B=1, project to 256x8x8).
    rng = t.Generator().manual_seed(1)
    z = t.randn(1, 256 * 8 * 8, generator=rng)
    feat = ex2_unflatten_to_feature_map(z, 256, 8, 8)
    assert feat.shape == (1, 256, 8, 8), f'decoder-shape: {tuple(feat.shape)}'
    # A different size assignment of the same flat input must produce a different shape.
    feat2 = ex2_unflatten_to_feature_map(z, 64, 16, 16)
    assert feat2.shape == (1, 64, 16, 16), 'must respect supplied named sizes'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_unflatten_to_feature_map(flat: Tensor, C: int, H: int, W: int) -> Tensor:
    return einops.rearrange(flat, 'b (c h w) -> b c h w', c=C, h=H, w=W)
```

**Why einops needs the named sizes.** When you flatten `'b c h w -> b (c h w)'` einops can infer the composite size from the input. Going the OTHER way, einops can't know how to split a single composite axis into three — `60 = 3·4·5` is one of many factorisations. You must supply `c=C, h=H, w=W` to disambiguate.

**Mapping order matters.** The leftmost axis in the parenthesis varies slowest. `'b (c h w) -> b c h w'` and `'b (w h c) -> b c h w'` produce DIFFERENT outputs from the same input — the latter is a transposed feature map. einops won't catch the bug; the row-major convention is on you.

**Difference from ex1.** ex1 was the forward flatten `'b c h w -> b (c h w)'` used to feed a CNN feature map into a `Linear` classifier head. ex2 is the inverse — the move you make in a decoder when going from a `Linear(latent → C·H·W)` projection BACK to a spatial feature map. Same op family, opposite direction.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()